# Notebook 6 – Bagging

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.

## Setup: Load & Split Data

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. Bagging
Short for **Bootstrap Aggregating**. Train the same algorithm on many random samples of the training data, then combine their predictions (vote/average). This reduces variance and makes the overall model more stable.

In [2]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
bagging = BaggingClassifier(DecisionTreeClassifier(random_state=42), n_estimators=50, random_state=42)
bagging.fit(X_train, y_train)
print("Bagging test acc:", accuracy_score(y_test, bagging.predict(X_test)))

Bagging test acc: 0.875


## 2. Bootstrap Sampling
Creating a new dataset by **randomly sampling rows with replacement** from the original data (same size, but some rows repeated, some left out). Each bagged model trains on a different bootstrap sample.

In [3]:
sample1 = df.sample(n=len(df), replace=True, random_state=1)
sample2 = df.sample(n=len(df), replace=True, random_state=2)
print("Sample 1 unique rows:", sample1.index.nunique(), "out of", len(df))
print("Sample 2 unique rows:", sample2.index.nunique(), "out of", len(df))

Sample 1 unique rows: 1892 out of 3000
Sample 2 unique rows: 1908 out of 3000


## 3. Bootstrap Aggregating
The full process: **bootstrap sample → train a model on each sample → aggregate (vote/average) their predictions**. This is what "Bagging" actually stands for and does internally.

In [4]:
print("Bagging uses bootstrap sampling by default:", bagging.bootstrap)

Bagging uses bootstrap sampling by default: True


## 4. Random Forest
A specific, very popular bagging method that uses Decision Trees as the base model, **and** adds extra randomness by only considering a random subset of features at each split (not just bootstrapped rows).

In [5]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print("Random Forest test acc:", accuracy_score(y_test, rf.predict(X_test)))

Random Forest test acc: 0.8733333333333333


## 5. Extra Trees
Short for **Extremely Randomized Trees**. Similar to Random Forest, but goes further: instead of finding the *best* split point for a feature, it picks a **random** split point. This adds even more randomness, usually trading a bit of accuracy for lower variance and faster training.

In [7]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et.fit(X_train, y_train)
print("Extra Trees test acc:", accuracy_score(y_test, et.predict(X_test)))

Extra Trees test acc: 0.8666666666666667


## 6. Out-of-Bag (OOB) Evaluation
Since bootstrap sampling leaves out roughly a third of the data for each tree, that leftover data ("out-of-bag") can be used to **evaluate the model without needing a separate validation set** — a free, built-in validation score.

In [8]:
rf_oob = RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True)
rf_oob.fit(X_train, y_train)
print("Out-of-Bag score:", rf_oob.oob_score_)
print("Test accuracy:      ", accuracy_score(y_test, rf_oob.predict(X_test)))

Out-of-Bag score: 0.89
Test accuracy:       0.8733333333333333


## Comparison: Decision Tree vs Random Forest vs Extra Trees

In [9]:
single_tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Extra Trees'],
    'Train Accuracy': [
        accuracy_score(y_train, single_tree.predict(X_train)),
        accuracy_score(y_train, rf.predict(X_train)),
        accuracy_score(y_train, et.predict(X_train)),
    ],
    'Test Accuracy': [
        accuracy_score(y_test, single_tree.predict(X_test)),
        accuracy_score(y_test, rf.predict(X_test)),
        accuracy_score(y_test, et.predict(X_test)),
    ]
})
comparison['Train-Test Gap'] = comparison['Train Accuracy'] - comparison['Test Accuracy']
comparison

,Model,Train Accuracy,Test Accuracy,Train-Test Gap
0,Decision Tree,0.915833,0.868333,0.047500
1,Random Forest,0.915833,0.873333,0.042500
2,Extra Trees,0.915833,0.866667,0.049167


## Why Ensemble Models Can Perform Better Than Individual Trees

- **A single Decision Tree** is prone to **overfitting** — it can grow very deep and memorize noise, leading to a large train-test gap (high variance).
- **Random Forest / Extra Trees average many trees**, each trained on different data (and, for Extra Trees, different random splits). Since each tree's errors are somewhat independent, averaging cancels much of that noise out — this **lowers variance without increasing bias much**.
- The result is usually a **smaller train-test gap** and **better, more stable test accuracy** than any single tree could achieve on its own — as seen in the table above.